<a href="https://colab.research.google.com/github/trialdyk/marchine-learning/blob/main/JS09/JS09_Tugas1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Praktikum Pembelajaran Mesin - JS09 Approximate Nearest Neighbors (ANN)
## Tugas Lab 1: Perbandingan ANNOY, FAISS, dan HNSW

Percobaan pada metric distance berbeda (Euclidean dan Cosine), jumlah data 1000 vs 1.000.000, dan dimensi 2D vs 5D.

### Langkah 1 - Instal Library

In [1]:
!pip install annoy faiss-cpu hnswlib

### Langkah 2 - Fungsi Benchmark

In [2]:
import numpy as np
import pandas as pd
import time
from annoy import AnnoyIndex
import faiss
import hnswlib

k = 10
n_query = 100
np.random.seed(42)

def exact_search(X, Q, metric):
    if metric == 'euclidean':
        idx = faiss.IndexFlatL2(X.shape[1])
        idx.add(X)
    else:
        Xn, Qn = X.copy(), Q.copy()
        faiss.normalize_L2(Xn)
        faiss.normalize_L2(Qn)
        idx = faiss.IndexFlatIP(X.shape[1])
        idx.add(Xn)
        Q = Qn
    return idx.search(Q, k)[1]

def recall(found, truth):
    return np.mean([len(set(f) & set(t)) / k for f, t in zip(found, truth)])

def run_annoy(X, Q, metric):
    dim = X.shape[1]
    idx = AnnoyIndex(dim, 'euclidean' if metric == 'euclidean' else 'angular')
    t = time.time()
    for i in range(len(X)):
        idx.add_item(i, X[i])
    idx.build(10)
    build = time.time() - t
    t = time.time()
    found = [idx.get_nns_by_vector(q, k) for q in Q]
    return build, (time.time() - t) / len(Q), found

def run_faiss(X, Q, metric):
    dim = X.shape[1]
    Xf, Qf = X.copy(), Q.copy()
    if metric == 'euclidean':
        idx = faiss.IndexFlatL2(dim)
    else:
        faiss.normalize_L2(Xf)
        faiss.normalize_L2(Qf)
        idx = faiss.IndexFlatIP(dim)
    t = time.time()
    idx.add(Xf)
    build = time.time() - t
    t = time.time()
    found = idx.search(Qf, k)[1]
    return build, (time.time() - t) / len(Q), found

def run_hnsw(X, Q, metric):
    dim = X.shape[1]
    idx = hnswlib.Index(space='l2' if metric == 'euclidean' else 'cosine', dim=dim)
    t = time.time()
    idx.init_index(max_elements=len(X), ef_construction=200, M=16)
    idx.add_items(X)
    build = time.time() - t
    idx.set_ef(50)
    t = time.time()
    found = idx.knn_query(Q, k=k)[0]
    return build, (time.time() - t) / len(Q), found

### Langkah 3 - Jalankan Percobaan

In [3]:
configs = [(1000, 2), (1000, 5), (1_000_000, 2), (1_000_000, 5)]
metrics = ['euclidean', 'cosine']
rows = []

for metric in metrics:
    for n_data, dim in configs:
        X = np.random.random((n_data, dim)).astype(np.float32)
        Q = np.random.random((n_query, dim)).astype(np.float32)
        truth = exact_search(X, Q, metric)
        for name, fn in [('ANNOY', run_annoy), ('FAISS', run_faiss), ('HNSW', run_hnsw)]:
            build, query, found = fn(X, Q, metric)
            rows.append({'Metric': metric, 'Jumlah Data/Dimensi': f'{n_data}/{dim}D', 'Algoritma': name,
                         'Build (detik)': build, 'Query (ms)': query * 1000, 'Recall@10': recall(found, truth)})
        print('selesai:', metric, n_data, dim)

hasil = pd.DataFrame(rows)
hasil

selesai: euclidean 1000 2
selesai: euclidean 1000 5


selesai: euclidean 1000000 2


selesai: euclidean 1000000 5
selesai: cosine 1000 2
selesai: cosine 1000 5


selesai: cosine 1000000 2


selesai: cosine 1000000 5


,Metric,Jumlah Data/Dimensi,Algoritma,Build (detik),Query (ms),Recall@10
0,euclidean,1000/2D,ANNOY,0.025925,0.027540,0.996
1,euclidean,1000/2D,FAISS,0.000040,0.005593,1.000
2,euclidean,1000/2D,HNSW,0.034700,0.012383,1.000
3,euclidean,1000/5D,ANNOY,0.018980,0.025325,0.963
4,euclidean,1000/5D,FAISS,0.000042,0.005300,1.000
5,euclidean,1000/5D,HNSW,0.041904,0.008414,1.000
6,euclidean,1000000/2D,ANNOY,12.017699,0.093777,1.000
7,euclidean,1000000/2D,FAISS,0.007634,1.994321,1.000
8,euclidean,1000000/2D,HNSW,53.407267,0.025640,1.000
9,euclidean,1000000/5D,ANNOY,8.648489,0.072758,0.958


### Langkah 4 - Tabel Performansi

Waktu build (detik)

In [4]:
hasil.pivot_table(index=['Metric', 'Jumlah Data/Dimensi'], columns='Algoritma', values='Build (detik)', sort=False).round(4)

Algoritma                        ANNOY   FAISS     HNSW
Metric    Jumlah Data/Dimensi                          
euclidean 1000/2D               0.0259  0.0000   0.0347
          1000/5D               0.0190  0.0000   0.0419
          1000000/2D           12.0177  0.0076  53.4073
          1000000/5D            8.6485  0.0155  71.5284
cosine    1000/2D               0.0366  0.0000   0.0283
          1000/5D               0.0199  0.0000   0.0307
          1000000/2D           33.0609  0.0031  62.2263
          1000000/5D           11.3505  0.0064  79.2931

Waktu query rata-rata per query (ms)

In [5]:
hasil.pivot_table(index=['Metric', 'Jumlah Data/Dimensi'], columns='Algoritma', values='Query (ms)', sort=False).round(4)

Algoritma                       ANNOY   FAISS    HNSW
Metric    Jumlah Data/Dimensi                        
euclidean 1000/2D              0.0275  0.0056  0.0124
          1000/5D              0.0253  0.0053  0.0084
          1000000/2D           0.0938  1.9943  0.0256
          1000000/5D           0.0728  2.1417  0.0293
cosine    1000/2D              0.0268  0.0044  0.0068
          1000/5D              0.0214  0.0034  0.0082
          1000000/2D           0.0721  2.2335  0.0348
          1000000/5D           0.1074  1.9129  0.0285

Recall@10 terhadap pencarian exact

In [6]:
hasil.pivot_table(index=['Metric', 'Jumlah Data/Dimensi'], columns='Algoritma', values='Recall@10', sort=False).round(4)

Algoritma                      ANNOY  FAISS   HNSW
Metric    Jumlah Data/Dimensi                     
euclidean 1000/2D              0.996    1.0  1.000
          1000/5D              0.963    1.0  1.000
          1000000/2D           1.000    1.0  1.000
          1000000/5D           0.958    1.0  1.000
cosine    1000/2D              0.999    1.0  1.000
          1000/5D              0.984    1.0  1.000
          1000000/2D           0.087    1.0  0.326
          1000000/5D           0.982    1.0  1.000

### Analisis

**1. Pengaruh jumlah data.** Pada 1.000 data, FAISS (exact) paling cepat untuk query (sekitar 0,003-0,006 ms) dan tidak perlu waktu build berarti. ANNOY dan HNSW lebih lambat (0,007-0,028 ms) karena overhead struktur index, sehingga tidak ada keuntungan memakai ANN pada data sekecil ini. Pada 1.000.000 data situasinya berbalik: query FAISS naik menjadi sekitar 1,9-2,2 ms karena memindai semua data, sedangkan HNSW tetap sekitar 0,026-0,035 ms (sekitar 60-80 kali lebih cepat) dan ANNOY sekitar 0,07-0,11 ms (sekitar 20-30 kali lebih cepat).

**2. Biaya build.** Percepatan query ANN dibayar dengan waktu build. Pada 1.000.000 data, build ANNOY 8,6-33 detik dan HNSW 53-79 detik, sedangkan FAISS Flat hanya 0,003-0,016 detik karena hanya menyalin vektor. HNSW paling mahal untuk dibangun tetapi paling cepat saat query; ANNOY di tengah.

**3. Pengaruh dimensi (2D vs 5D).** Perbedaan antara 2D dan 5D pada percobaan ini kecil untuk waktu query. Dimensi yang lebih tinggi sedikit menurunkan recall ANNOY pada Euclidean (1.000 data: 0,996 menjadi 0,963; 1.000.000 data: 1,000 menjadi 0,958), sedangkan HNSW tetap 1,0. Waktu build HNSW sedikit naik pada 5D (53 detik menjadi 72 detik pada Euclidean).

**4. Pengaruh metric.** Euclidean dan Cosine menghasilkan pola waktu yang mirip. Perbedaan menonjol ada pada kasus Cosine 1.000.000 data 2D: recall ANNOY hanya 0,087 dan HNSW 0,326. Pada data 2D yang tersebar acak di kuadran positif, Cosine hanya melihat sudut, sehingga satu juta titik menumpuk pada sudut yang hampir sama dan urutan 10 tetangga terdekat menjadi hampir acak (banyak nilai kemiripan yang nyaris sama). Jadi nilai recall yang rendah di sini lebih mencerminkan ketidakstabilan ground truth daripada kelemahan algoritma. Pada Cosine 5D recall kembali tinggi (ANNOY 0,982; HNSW 1,0).

**5. Akurasi.** FAISS Flat selalu memiliki recall 1,0 karena merupakan pencarian exact. HNSW memiliki recall 1,0 pada hampir semua kombinasi, kecuali kasus Cosine 2D 1.000.000 di atas. ANNOY memiliki recall 0,96-1,0 pada kasus normal, sedikit lebih rendah daripada HNSW dengan parameter yang dipakai (10 tree).

**Kesimpulan penggunaan.** Untuk data kecil gunakan exact (FAISS Flat). Untuk data besar dengan query berulang, HNSW memberi kecepatan query dan akurasi terbaik jika waktu build yang lama dapat diterima; ANNOY memberi build yang lebih cepat dengan akurasi sedikit lebih rendah.